# Experiment 6


In [ ]:
import re
import math
import random
from collections import Counter, defaultdict
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 
random.seed(42)
np.random.seed(42)

In [ ]:
# Step 1: read and pre-process the corpus
with open("glove_corpus.txt", encoding="utf-8") as f:
    raw_text = f.read()
 
STOP_WORDS = {"the", "a", "an", "is", "in", "of", "and", "at", "to"}
 
sentences = []
for line in raw_text.lower().splitlines():
    tokens = re.findall(r"[a-z]+", line)
    tokens = [t for t in tokens if t not in STOP_WORDS]
    if tokens:
        sentences.append(tokens)
 
print("Number of sentences:", len(sentences))
for s in sentences[:5]:
    print(s)

In [ ]:
# Step 2: vocabulary + co-occurrence matrix
word_counts = Counter(w for s in sentences for w in s)
vocab = sorted(word_counts)
word2id = {w: i for i, w in enumerate(vocab)}
id2word = {i: w for w, i in word2id.items()}
V = len(vocab)
print("Vocabulary size:", V)
 
WINDOW = 4
X = np.zeros((V, V), dtype=np.float64)
for s in sentences:
    ids = [word2id[w] for w in s]
    for c, wi in enumerate(ids):
        for d in range(1, WINDOW + 1):
            if c + d < len(ids):
                wj = ids[c + d]
                X[wi, wj] += 1.0 / d   # distance weighting
                X[wj, wi] += 1.0 / d   # symmetric
 
print("Non-zero co-occurrence entries:", int((X > 0).sum()))
pd.DataFrame(X, index=vocab, columns=vocab).loc[
    ["king", "queen", "man", "woman"], ["rules", "kingdom", "palace", "royal", "strong"]
].round(2)

In [ ]:
# Steps 3 & 4: train GloVe with AdaGrad
class GloVe:
    def __init__(self, X, dim=50, x_max=10.0, alpha=0.75, lr=0.05, seed=42):
        rng = np.random.default_rng(seed)
        self.V = X.shape[0]
        self.rows, self.cols = np.nonzero(X)
        self.xij = X[self.rows, self.cols]
        self.log_x = np.log(self.xij)
        self.weights = np.minimum((self.xij / x_max) ** alpha, 1.0)
        self.W = (rng.random((self.V, dim)) - 0.5) / dim        # main vectors
        self.W_ctx = (rng.random((self.V, dim)) - 0.5) / dim    # context vectors
        self.b = np.zeros(self.V)
        self.b_ctx = np.zeros(self.V)
        # AdaGrad accumulators
        self.gW, self.gWc = np.ones_like(self.W), np.ones_like(self.W_ctx)
        self.gb, self.gbc = np.ones(self.V), np.ones(self.V)
        self.lr = lr
 
    def train(self, epochs=500, verbose_every=100):
        history = []
        idx = np.arange(len(self.xij))
        for epoch in range(1, epochs + 1):
            np.random.shuffle(idx)
            total = 0.0
            for k in idx:
                i, j = self.rows[k], self.cols[k]
                diff = self.W[i] @ self.W_ctx[j] + self.b[i] + self.b_ctx[j] - self.log_x[k]
                fdiff = self.weights[k] * diff
                total += 0.5 * fdiff * diff
                grad_wi = fdiff * self.W_ctx[j]
                grad_wj = fdiff * self.W[i]
                self.W[i] -= self.lr * grad_wi / np.sqrt(self.gW[i])
                self.W_ctx[j] -= self.lr * grad_wj / np.sqrt(self.gWc[j])
                self.b[i] -= self.lr * fdiff / np.sqrt(self.gb[i])
                self.b_ctx[j] -= self.lr * fdiff / np.sqrt(self.gbc[j])
                self.gW[i] += grad_wi ** 2
                self.gWc[j] += grad_wj ** 2
                self.gb[i] += fdiff ** 2
                self.gbc[j] += fdiff ** 2
            history.append(total)
            if epoch % verbose_every == 0 or epoch == 1:
                print(f"Epoch {epoch:4d} | loss = {total:.4f}")
        return history
 
    @property
    def embeddings(self):
        return self.W + self.W_ctx
 
 
glove = GloVe(X, dim=50, lr=0.05)
loss_history = glove.train(epochs=500, verbose_every=100)
 
plt.figure(figsize=(6, 3))
plt.plot(loss_history)
plt.xlabel("Epoch"); plt.ylabel("Weighted least-squares loss")
plt.title("GloVe (from scratch) training loss")
plt.tight_layout(); plt.show()

In [ ]:
# Step 5: cosine-similarity search
E = glove.embeddings
E_norm = E / np.linalg.norm(E, axis=1, keepdims=True)
 
def most_similar(word, topn=5):
    v = E_norm[word2id[word]]
    sims = E_norm @ v
    order = np.argsort(-sims)
    return [(id2word[i], float(sims[i])) for i in order if id2word[i] != word][:topn]
 
print("Vector for 'king' (first 10 dims):\n", np.round(E[word2id["king"]][:10], 4))
print("\nTop-5 words most similar to 'king' (GloVe from scratch):")
pd.DataFrame(most_similar("king", 5), columns=["word", "cosine_similarity"])

In [ ]:
for w in ["queen", "man", "cat", "computer"]:
    print(f"{w:9s} ->", [(x, round(s, 3)) for x, s in most_similar(w, 3)])

In [ ]:
# 2-D visualisation of the learned embeddings (PCA via SVD)
centered = E - E.mean(axis=0)
U, S, Vt = np.linalg.svd(centered, full_matrices=False)
coords = centered @ Vt[:2].T
 
show = ["king", "queen", "prince", "princess", "man", "woman", "kingdom", "palace",
        "cat", "dog", "milk", "computer", "software", "data", "sun", "moon", "earth"]
plt.figure(figsize=(7, 5))
for w in show:
    x, y = coords[word2id[w]]
    plt.scatter(x, y, color="tab:blue")
    plt.annotate(w, (x, y), textcoords="offset points", xytext=(4, 3))
plt.title("GloVe (from scratch) embeddings – PCA projection")
plt.tight_layout(); plt.show()

In [ ]:
import gensim.downloader as api
 
glove_pretrained = api.load("glove-wiki-gigaword-100")   # KeyedVectors
print("Vocabulary size :", len(glove_pretrained.key_to_index))
print("Vector dimension:", glove_pretrained.vector_size)
print("\nVector for 'king' (first 10 dims):\n", glove_pretrained["king"][:10])

In [ ]:
print("Top-10 words most similar to 'king' (pretrained GloVe):")
pd.DataFrame(glove_pretrained.most_similar("king", topn=10), columns=["word", "cosine_similarity"])

In [ ]:
print("Analogy  king - man + woman =", glove_pretrained.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
for a, b in [("king", "queen"), ("king", "man"), ("king", "cat")]:
    print(f"similarity({a}, {b}) = {glove_pretrained.similarity(a, b):.4f}")
 
# Compare with words that also exist in our small corpus
common = [w for w in ["queen", "prince", "kingdom", "palace", "man", "woman", "cat", "computer"] if w in word2id]
pd.DataFrame({
    "word": common,
    "scratch_GloVe_sim": [round(float(E_norm[word2id['king']] @ E_norm[word2id[w]]), 4) for w in common],
    "pretrained_GloVe_sim": [round(float(glove_pretrained.similarity("king", w)), 4) for w in common],
})

In [ ]:
with open("ngram_corpus.txt", encoding="utf-8") as f:
    corpus = f.read().lower()
 
ngram_sentences = [re.findall(r"[a-z]+", s) for s in corpus.split(".") if s.strip()]
for i, s in enumerate(ngram_sentences, 1):
    print(i, " ".join(s))

In [ ]:
class NGramLanguageModel:
    def __init__(self, n=2):
        self.n = n
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.next_words = defaultdict(Counter)
        self.vocab = set()
 
    def _pad(self, tokens):
        return ["<s>"] * (self.n - 1) + tokens + ["</s>"]
 
    def fit(self, sentences):
        for tokens in sentences:
            padded = self._pad(tokens)
            self.vocab.update(padded)
            for i in range(len(padded) - self.n + 1):
                gram = tuple(padded[i:i + self.n])
                ctx, word = gram[:-1], gram[-1]
                self.ngram_counts[gram] += 1
                self.context_counts[ctx] += 1
                self.next_words[ctx][word] += 1
        self.vocab.discard("<s>")
        return self
 
    def prob(self, context, word, smoothing="mle"):
        ctx = tuple(context[-(self.n - 1):]) if self.n > 1 else ()
        c_gram = self.ngram_counts[ctx + (word,)]
        c_ctx = self.context_counts[ctx]
        if smoothing == "laplace":
            return (c_gram + 1) / (c_ctx + len(self.vocab))
        return c_gram / c_ctx if c_ctx else 0.0
 
    def predict_next(self, context, k=3):
        ctx = tuple(context[-(self.n - 1):]) if self.n > 1 else ()
        total = self.context_counts[ctx]
        return [(w, c / total) for w, c in self.next_words[ctx].most_common(k)] if total else []
 
    def sentence_prob(self, tokens, smoothing="laplace"):
        padded = self._pad(tokens)
        log_p = 0.0
        for i in range(self.n - 1, len(padded)):
            p = self.prob(padded[:i], padded[i], smoothing)
            if p == 0:
                return 0.0, float("inf")
            log_p += math.log(p)
        m = len(padded) - (self.n - 1)
        return math.exp(log_p), math.exp(-log_p / m)   # probability, perplexity
 
 
models = {n: NGramLanguageModel(n).fit(ngram_sentences) for n in (1, 2, 3)}
for n, m in models.items():
    print(f"{n}-gram model: {len(m.ngram_counts):3d} distinct n-grams, |V| = {len(m.vocab)}")

In [ ]:
# Most frequent n-grams
for n in (1, 2, 3):
    top = models[n].ngram_counts.most_common(8)
    print(f"Top {n}-grams:", [(" ".join(g), c) for g, c in top])

In [ ]:
# Bigram probability table for a few words
bi = models[2]
rows = ["natural", "language", "machine", "deep", "artificial"]
cols = ["language", "processing", "learning", "intelligence", "models", "is"]
pd.DataFrame([[round(bi.prob([r], c), 3) for c in cols] for r in rows], index=rows, columns=cols)

In [ ]:
def predict_with_backoff(prefix, k=3):
    tokens = ["<s>", "<s>"] + re.findall(r"[a-z]+", prefix.lower())
    for n in (3, 2, 1):
        preds = models[n].predict_next(tokens, k)
        if preds:
            return n, preds
    return 0, []
 
for prefix in ["natural language", "machine", "deep learning is", "artificial intelligence",
               "language", "a language model predicts the", "computers"]:
    order, preds = predict_with_backoff(prefix)
    pretty = ", ".join(f"{w} ({p:.2f})" for w, p in preds)
    print(f"'{prefix}' -> [{order}-gram] {pretty}")

In [ ]:
test_sentences = [
    "natural language processing is a branch of artificial intelligence",
    "deep learning is a branch of artificial intelligence",
    "machine learning enables computers to understand human language",
    "computers understand intelligence deep",
]
rows = []
for s in test_sentences:
    toks = s.split()
    row = {"sentence": s}
    for n in (2, 3):
        p, pp = models[n].sentence_prob(toks, smoothing="laplace")
        row[f"{n}-gram P"] = f"{p:.2e}"
        row[f"{n}-gram PPL"] = round(pp, 2)
    rows.append(row)
pd.DataFrame(rows)

In [ ]:
def generate(n=3, seed_words="", max_len=15, greedy=False, seed=0):
    rng = random.Random(seed)
    model = models[n]
    tokens = ["<s>"] * (n - 1) + seed_words.split()
    while len(tokens) < max_len + n:
        preds = model.next_words[tuple(tokens[-(n - 1):])]
        if not preds:
            break
        if greedy:
            word = preds.most_common(1)[0][0]
        else:
            words, counts = zip(*preds.items())
            word = rng.choices(words, weights=counts)[0]
        if word == "</s>":
            break
        tokens.append(word)
    return " ".join(t for t in tokens if t != "<s>")
 
print("Greedy (trigram)  :", generate(3, "natural", greedy=True))
for i in range(3):
    print(f"Sampled (trigram) {i+1}:", generate(3, "", seed=i))
print("Sampled (bigram)  :", generate(2, "deep", seed=7))